# Download Placebo Assets (CCData) — XRP & DOGE

**Purpose:** Download hourly spot data for non-DeFi-collateral assets used as placebo controls in the robustness tests (notebook 08).

**Rationale:** XRP and DOGE have zero meaningful presence as collateral in Ethereum DeFi lending protocols. If DeFi liquidation shocks affect these assets as strongly as ETH, the endogenous amplification story is dead. If ETH shows excess sensitivity relative to these placebos, the liquidation channel is confirmed.

**Source:** CCData (CryptoCompare CCCAGG) — same source as BTC/ETH spot benchmarks.
**Window:** [2021-03-15, 2025-12-01) UTC — matches the master calendar.

In [18]:
import requests
import pandas as pd
import time
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ── CONFIG ──
API_KEY = "YOUR_API_KEY"  # ← Replace with your CCData API key

SYMBOLS = ["XRP", "DOGE"]
CURRENCY = "USD"
START_DATE = "2021-03-15 00:00:00"
END_DATE = "2025-12-01 00:00:00"
LIMIT = 2000

# Output: directly to normalized/spot/ (analysis-ready)
OUT_DIR = Path("..") / ".." / ".." / "data" / "normalized" / "spot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── ROBUST SESSION ──
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

print(f"Output dir: {OUT_DIR.resolve()}")
print(f"Symbols: {SYMBOLS}")
print(f"Window: [{START_DATE}, {END_DATE})")

Output dir: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage/data/normalized/spot
Symbols: ['XRP', 'DOGE']
Window: [2021-03-15 00:00:00, 2025-12-01 00:00:00)


In [20]:
def download_ccdata_histohour(symbol, start_date, end_date, api_key):
    print(f"\n📥 Downloading CCData (CCCAGG) for {symbol}/USD...")

    start_ts = int(pd.Timestamp(start_date, tz='UTC').timestamp())
    end_ts = int(pd.Timestamp(end_date, tz='UTC').timestamp())

    all_data = []
    current_to_ts = end_ts
    headers = {"authorization": f"Apikey {api_key}"}

    while current_to_ts >= start_ts:
        url = f"https://min-api.cryptocompare.com/data/v2/histohour?fsym={symbol}&tsym={CURRENCY}&limit={LIMIT}&toTs={current_to_ts}"

        try:
            response = session.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                print(f"❌ API error: {response.status_code}. Retrying in 2s...")
                time.sleep(2)
                continue
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Connection issue. Retrying in 3s...")
            time.sleep(3)
            continue

        json_data = response.json()
        if json_data.get("Response") == "Error":
            print(f"❌ CCData error: {json_data.get('Message')}")
            break

        data = json_data.get("Data", {}).get("Data", [])
        if not data:
            break

        all_data.extend(data)
        oldest_ts = data[0]["time"]

        if oldest_ts <= start_ts:
            break

        current_to_ts = oldest_ts - 3600
        time.sleep(0.5)
        print(f"  ← fetched back to {pd.to_datetime(oldest_ts, unit='s', utc=True)}")

    if not all_data:
        print(f"⚠️ No data for {symbol}!")
        return None

    df = pd.DataFrame(all_data)
    df['date'] = pd.to_datetime(df['time'], unit='s', utc=True)

    if 'volumefrom' in df.columns:
        df = df.rename(columns={'volumefrom': 'volume'})

    df = df[['date', 'open', 'high', 'low', 'close', 'volume']]
    df = df.sort_values('date').drop_duplicates('date')
    df = df[(df['date'] >= pd.Timestamp(start_date, tz='UTC')) &
            (df['date'] < pd.Timestamp(end_date, tz='UTC'))]

    output_file = OUT_DIR / f"{symbol.lower()}_ccdata_1h.parquet"
    df.to_parquet(output_file, index=False, engine="pyarrow")
    print(f"✅ Done! {len(df):,} rows saved to {output_file}")

    # Quick QA
    expected = 41328  # hours in [2021-03-15, 2025-12-01)
    missing = expected - len(df)
    print(f"   Expected: {expected:,} | Got: {len(df):,} | Missing: {missing}")
    if missing > 100:
        print(f"   ⚠️ {missing} missing hours — check coverage")

    return df

In [22]:
# ── DOWNLOAD ──
if API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ Replace API_KEY with your CCData key before running!")
else:
    results = {}
    for sym in SYMBOLS:
        results[sym] = download_ccdata_histohour(sym, START_DATE, END_DATE, API_KEY)

    print("\n" + "=" * 55)
    print("SUMMARY")
    print("=" * 55)
    for sym, df in results.items():
        if df is not None:
            print(f"  {sym}: {len(df):,} rows, [{df['date'].min()}, {df['date'].max()}]")
        else:
            print(f"  {sym}: FAILED")


📥 Downloading CCData (CCCAGG) for XRP/USD...
  ← fetched back to 2025-09-08 16:00:00+00:00
  ← fetched back to 2025-06-17 07:00:00+00:00
  ← fetched back to 2025-03-25 22:00:00+00:00
  ← fetched back to 2025-01-01 13:00:00+00:00
  ← fetched back to 2024-10-10 04:00:00+00:00
  ← fetched back to 2024-07-18 19:00:00+00:00
  ← fetched back to 2024-04-26 10:00:00+00:00
  ← fetched back to 2024-02-03 01:00:00+00:00
  ← fetched back to 2023-11-11 16:00:00+00:00
  ← fetched back to 2023-08-20 07:00:00+00:00
  ← fetched back to 2023-05-28 22:00:00+00:00
  ← fetched back to 2023-03-06 13:00:00+00:00
  ← fetched back to 2022-12-13 04:00:00+00:00
  ← fetched back to 2022-09-20 19:00:00+00:00
  ← fetched back to 2022-06-29 10:00:00+00:00
  ← fetched back to 2022-04-07 01:00:00+00:00
  ← fetched back to 2022-01-13 16:00:00+00:00
  ← fetched back to 2021-10-22 07:00:00+00:00
  ← fetched back to 2021-07-30 22:00:00+00:00
  ← fetched back to 2021-05-08 13:00:00+00:00
✅ Done! 41,328 rows saved to ../..

In [26]:
# ── VERIFICATION ──
import sys; sys.path.insert(0, "../../..")
from config import CFG

for sym in ["xrp", "doge"]:
    path = CFG.ROOT / "data" / "normalized" / "spot" / f"{sym}_ccdata_1h.parquet"
    if path.exists():
        df = pd.read_parquet(path, engine="pyarrow")
        df["date"] = pd.to_datetime(df["date"], utc=True)
        print(f"{sym.upper()}: {len(df):,} rows, [{df['date'].min()}, {df['date'].max()}]")
    else:
        print(f"{sym.upper()}: file not found at {path}")

print("\n✅ Placebo download complete")

XRP: 41,328 rows, [2021-03-15 00:00:00+00:00, 2025-11-30 23:00:00+00:00]
DOGE: 41,328 rows, [2021-03-15 00:00:00+00:00, 2025-11-30 23:00:00+00:00]

✅ Placebo download complete
